In [22]:
import os
import plotly.express as px
import plotly.graph_objects as go

import pandas as pd
import pycountry
os.getcwd()

data_dir='../../output/'

def get_iso_codes(name):
    '''get iso codes for map visualization'''
    try:
        name = name.replace("_", " ")
        country = pycountry.countries.search_fuzzy(name)[0]
        return pd.Series({"ISO2": country.alpha_2, "ISO3": country.alpha_3})
    except:
        return pd.Series({"ISO2": None, "ISO3": None})
    
def plot_total_deaths_per_country(df):
    df=add_iso_codes(df)

    # df[["ISO2", "ISO3"]] = df["_id"].apply(get_iso_codes)
    
    fig = go.Figure(go.Choropleth(
        locations=df["ISO3"],
        z=df["percTotalDeaths"],
        text=df["geoId"], 
        hovertemplate='<b>%{text}</b><br>Deaths: %{customdata[0]}<extra></extra>',
        customdata=df[["totalDeaths"]],
        colorscale="Plasma",
        colorbar_title="%(Deaths)"
    ))

    fig.update_layout(
        title_text="% COVID-19 Deaths by Country",
        margin=dict(l=0, r=0, t=50, b=0)
    )

    fig.update_geos(
        showcoastlines=True,
        showland=True,
        showframe=False,
        fitbounds="locations"             
    )

    fig.show()
    return fig

def plot_total_cases_per_country(df):
    df=add_iso_codes(df)

    # df[["ISO2", "ISO3"]] = df["_id"].apply(get_iso_codes)
    
    fig = go.Figure(go.Choropleth(
        locations=df["ISO3"],
        z=df["percTotalCases"],
        text=df["geoId"], 
        hovertemplate='<b>%{text}</b><br>Cases: %{customdata[0]}<extra></extra>',
        customdata=df[["totalCases"]],
        colorscale="viridis",
        colorbar_title="%(Cases)"
    ))

    fig.update_layout(
        title_text="% Cases by Country",
        margin=dict(l=0, r=0, t=50, b=0)
    )

    fig.update_geos(
        showcoastlines=True,
        showland=True,
        showframe=False,
        fitbounds="locations"             
    )

    fig.show()
    return fig

def plot_avg_incidence_per_country(df):
    df=add_iso_codes(df)

    # df[["ISO2", "ISO3"]] = df["_id"].apply(get_iso_codes)
    
    fig = go.Figure(go.Choropleth(
        locations=df["ISO3"],
        z=df["avg14DayIncidence"],
        text=df["geoId"], 
        hovertemplate='<b>%{text}</b><br>Avg Incidence: %{customdata[0]}<extra></extra>',
        customdata=df[["avg14DayIncidence"]],
        colorscale="Cividis",
        colorbar_title="Avg 14 day Incidence"
    ))

    fig.update_layout(
        title_text="Average COVID-19 Incidence by Country",
        margin=dict(l=0, r=0, t=50, b=0)
    )

    fig.update_geos(
        showcoastlines=True,
        showland=True,
        showframe=False,
        fitbounds="locations"             
    )

    fig.show()
    return fig

def add_iso_codes(df):
    iso_map = {}
    for name in df["_id"].unique():
        try:
            clean_name = name.replace("_", " ")
            country = pycountry.countries.search_fuzzy(clean_name)[0]
            iso_map[name] = (country.alpha_2, country.alpha_3)
        except:
            iso_map[name] = (None, None)

    df["ISO2"] = df["_id"].map(lambda x: iso_map[x][0])
    df["ISO3"] = df["_id"].map(lambda x: iso_map[x][1])
    return df

In [37]:
mongo_file=data_dir+'mongodb/country_stats.json'
output='../../figures/mongodb/'
df=pd.read_json(mongo_file)
df.columns, df['_id']

(Index(['_id', 'geoId', 'totalCases', 'totalDeaths', 'popData2019',
        'avg14DayIncidence', 'percTotalCases', 'percTotalDeaths'],
       dtype='object'),
 0                                 Kenya
 1                               Germany
 2                         Faroe_Islands
 3                            Montenegro
 4                             Palestine
                      ...               
 209                            Zimbabwe
 210                        Cote_dIvoire
 211    Saint_Vincent_and_the_Grenadines
 212                              Guinea
 213                       Guinea_Bissau
 Name: _id, Length: 214, dtype: object)

In [38]:
df=add_iso_codes(df)

fig1=plot_total_deaths_per_country(df)
fig2=plot_total_cases_per_country(df)
fig3=plot_avg_incidence_per_country(df)

In [39]:
df2=pd.read_csv(data_dir+'hadoop/country_stats.csv',on_bad_lines='warn')


# Index(['_id', 'geoId', 'totalCases', 'totalDeaths', 'popData2019',
    #    'avg14DayIncidence', 'percTotalCases', 'percTotalDeaths'],
    #   dtype='object')

# df2['avg14DayIncidence']=df2["Cumulative_number_for_14_days_of_COVID-19_cases_per_100000"]
# df2['percTotalCases']=df2["cases"] / df2["popData2019"] * 100
# df2['percTotalDeaths']=df2["deaths"] / df2["popData2019"] * 100
# df2['totalCases']=df2["cases"]
# df2['totalDeaths']=df2["deaths"]
df2["_id"]=df2["geoId"]
# # df2.columns, df.columns

# df2[["ISO2", "ISO3"]] = df2["_id"].apply(get_iso_codes)
df2.columns

# df2=add_iso_codes(df2)

# df2.columns

# check which country has highest percTotalDeaths in df2
# max_perc_deaths_country = df2.loc[df2['percTotalDeaths'].idxmax()]
# print(f"Country with highest % of total deaths in df2: {max_perc_deaths_country['countriesAndTerritories']} with {max_perc_deaths_country['totalDeaths']:.2f}%")  
# # same for total cases
# max_perc_cases_country = df2.loc[df2['percTotalCases'].idxmax()]
# print(f"Country with highest % of total cases in df2: {max_perc_cases_country['countriesAndTerritories']} with {max_perc_cases_country['totalCases']:.2f}%")  
# # same for avg14DayIncidence
# max_avg_incidence_country = df2.loc[df2['avg14DayIncidence'].idxmax()]
# print(f"Country with highest avg 14 day incidence in df2: {max_avg_incidence_country['countriesAndTerritories']} with {max_avg_incidence_country['avg14DayIncidence']:.2f}")

# df2['popData2019']

/tmp/ipykernel_5106/1710850313.py:1: ParserWarning:

Skipping line 26: expected 8 fields, saw 9




Index(['countriesAndTerritories', 'geoId', 'totalCases', 'totalDeaths',
       'popData2019', 'avg14DayIncidence', 'percTotalCases', 'percTotalDeaths',
       '_id'],
      dtype='object')

In [35]:
fig4=plot_total_deaths_per_country(df2)
fig5=plot_total_cases_per_country(df2)
fig6=plot_avg_incidence_per_country(df2)